# Training

In [ ]:
%reset -f
import torch
import torch.nn as nn
from torchinfo import summary
import joblib

from nn_pytorch.data import load_data, split_data, scale_data, make_loaders
import nn_pytorch.config as config
from nn_pytorch.models.FNN import FNN
from nn_pytorch.trainer import train_one_epoch, evaluate
from nn_pytorch.utils.device import get_device
from nn_pytorch.plots import save_loss_plot


In [ ]:
torch.manual_seed(config.RANDOM_STATE)
DEVICE = get_device()
print(f'Using device: {DEVICE}')

X, y = load_data(config.DATA_DIR/'data.csv')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, random_state=config.RANDOM_STATE)
X_train_s, X_val_s, X_test_s, y_train_s, y_val_s, y_test_s, sX, sY = scale_data(X_train, X_val, X_test, y_train, y_val, y_test)
train_loader, val_loader = make_loaders(X_train_s, X_val_s, y_train_s, y_val_s, batch_size=config.BATCH_SIZE, device=DEVICE)

scalers = {'scaler_X': sX,
            'scaler_Y': sY}
joblib.dump(scalers, config.MODEL_DIR/'scalers.pkl')

in_feat = X_train_s.shape[1]
out_feat = y_train_s.shape[1]


mdl = FNN(in_feat, config.H1, config.H2, out_feat).to(DEVICE)
summary(mdl, input_size=(config.BATCH_SIZE, in_feat), device=DEVICE)

criterion = nn.MSELoss()
optimiser = torch.optim.Adam(mdl.parameters(), lr=config.LR)

train_losses = []
val_losses = []

best_val_loss = float("inf")
counter = 0

for epoch in range(config.EPOCHS):
    train_loss = train_one_epoch(mdl, train_loader, criterion, optimiser)
    val_loss = evaluate(mdl, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | train={train_loss:.4f} | val={val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss     
        counter = 0
        checkpoint = {
            'model_state_dict': mdl.state_dict(),
            'in_feat': in_feat,
            'out_feat': out_feat,
            'h1': config.H1,
            'h2': config.H2}
        torch.save(checkpoint, config.MODEL_DIR/'checkpoint.pth')
    else:
        counter += 1
        if counter >= config.PATIENCE:
            print(f"Early stopping at epoch: {epoch}")
            break

save_loss_plot(train_losses, val_losses, config.FIG_DIR/'loss.pdf')
print(f"plots saved to {config.FIG_DIR}")

# Evaluation

In [ ]:
%reset -f
import torch
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

import nn_pytorch.config as config
from nn_pytorch.data import load_data, split_data
from nn_pytorch.models.FNN import FNN
from nn_pytorch.plots import save_parity_plot
from nn_pytorch.utils.device import get_device


In [ ]:
DEVICE = get_device()
print(f'Using device: {DEVICE}')

# load checkpoint
checkpoint = torch.load(config.MODEL_DIR/'checkpoint.pth', map_location=DEVICE)

# load scalers
scalers = joblib.load(config.MODEL_DIR/'scalers.pkl')
sX = scalers['scaler_X']
sY = scalers['scaler_Y']

# data
X, y = load_data(config.DATA_DIR/'data.csv')
_, _, X_test, _, _, y_test = split_data(X, y, random_state=config.RANDOM_STATE)
X_test_s = sX.transform(X_test)
y_test_s = sY.transform(y_test.reshape(-1, 1))

# load model
mdl = FNN(checkpoint['in_feat'], checkpoint['h1'], checkpoint['h2'], checkpoint['out_feat']).to(DEVICE)
mdl.load_state_dict(checkpoint['model_state_dict'])
mdl.eval()
print('model loaded successfully')

X_test_t = torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)
with torch.inference_mode():
    preds_s = mdl(X_test_t).cpu().numpy()

# inverse transform
y_true = sY.inverse_transform(y_test_s).flatten()
y_pred = sY.inverse_transform(preds_s).flatten()

# metric
rmse = root_mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"\n{'='*40}")
print(f'  Test set evaluation')
print(f"{'='*40}")
print(f"  RMSE : {rmse:>12,.2f}")
print(f"  MAE  : {mae:>12,.2f}")
print(f"  R²   : {r2:>12.4f}")
print(f"{'='*40}\n")

# plots
save_parity_plot(y_true, y_pred, config.FIG_DIR/'parity.pdf')
print(f"plots saved to {config.FIG_DIR}")